In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Krzychu\Documents\Python\Phishing_URL_detector\data\phishing_url_dataset_unique.csv')

df


,url,label,source
0,http://110.37.26.193:54956/bin.sh,1,urlhaus
1,https://sentry5.obor1shwron8.ru/4ylkytvt,1,urlhaus
2,https://d6gu.ripple-cask.ru/jid43vpd,1,urlhaus
3,http://130.12.180.34/596a96cc7bf9108cd896f33c4...,1,urlhaus
4,https://bronze.systems,0,tranco
...,...,...,...
48807,https://zelnip.mo5hnap2sser.ru/b1a0bitj,1,urlhaus
48808,https://ittefaq.com.bd,0,tranco
48809,https://atidevs.com,0,tranco
48810,http://45.156.87.115/hiddenbin/boatnet.spc,1,urlhaus


In [2]:
df["url_length"] = df["url"].str.len()
df[["url", "url_length", "label"]].head()



,url,url_length,label
0,http://110.37.26.193:54956/bin.sh,33,1
1,https://sentry5.obor1shwron8.ru/4ylkytvt,40,1
2,https://d6gu.ripple-cask.ru/jid43vpd,36,1
3,http://130.12.180.34/596a96cc7bf9108cd896f33c4...,91,1
4,https://bronze.systems,22,0


In [3]:
df.groupby("label")["url_length"].mean()

label
0    20.025035
1    36.128247
Name: url_length, dtype: float64

In [4]:
df["dot_count"] = df["url"].str.count(r"\.")
df.groupby("label")["dot_count"].mean()

label
0    1.074859
1    3.177661
Name: dot_count, dtype: float64

In [5]:
df["hyphen_count"] = df["url"].str.count("-")
df.groupby("label")["hyphen_count"].mean()

label
0    0.080677
1    0.136524
Name: hyphen_count, dtype: float64

In [6]:
df["digit_count"] = df["url"].str.count(r"\d")
df.groupby("label")["digit_count"].mean()

label
0     0.142383
1    11.544333
Name: digit_count, dtype: float64

In [7]:
df["has_https"] = df["url"].str.startswith("https").astype(int)
df
#df.groupby("label")["has_https"].mean()

,url,label,source,url_length,dot_count,hyphen_count,digit_count,has_https
0,http://110.37.26.193:54956/bin.sh,1,urlhaus,33,4,0,15,0
1,https://sentry5.obor1shwron8.ru/4ylkytvt,1,urlhaus,40,2,0,4,1
2,https://d6gu.ripple-cask.ru/jid43vpd,1,urlhaus,36,2,1,3,1
3,http://130.12.180.34/596a96cc7bf9108cd896f33c4...,1,urlhaus,91,4,0,46,0
4,https://bronze.systems,0,tranco,22,1,0,0,1
...,...,...,...,...,...,...,...,...
48807,https://zelnip.mo5hnap2sser.ru/b1a0bitj,1,urlhaus,39,2,0,4,1
48808,https://ittefaq.com.bd,0,tranco,22,2,0,0,1
48809,https://atidevs.com,0,tranco,19,1,0,0,1
48810,http://45.156.87.115/hiddenbin/boatnet.spc,1,urlhaus,42,4,0,10,0


In [8]:
import re

def has_ip(url):
    if re.search(r"https?://\d+\.\d+\.\d+\.\d+", url):
        return 1
    else:
        return 0

df["has_ip"] = df["url"].apply(has_ip)
df.groupby("label")["has_ip"].mean()

label
0    0.000000
1    0.711219
Name: has_ip, dtype: float64

In [9]:
def has_suspicious(url):
    suspicious = ["login", "verify", "secure", "account", "bank", "update"]
    url_lower = url.lower()
    for word in suspicious:
        if word in url_lower:
            return 1
    return 0

df["has_suspicious"] = df["url"].apply(has_suspicious)
df.groupby("label")["has_suspicious"].mean()        


label
0    0.005163
1    0.003974
Name: has_suspicious, dtype: float64

In [10]:
df.head()

,url,label,source,url_length,dot_count,hyphen_count,digit_count,has_https,has_ip,has_suspicious
0,http://110.37.26.193:54956/bin.sh,1,urlhaus,33,4,0,15,0,1,0
1,https://sentry5.obor1shwron8.ru/4ylkytvt,1,urlhaus,40,2,0,4,1,0,0
2,https://d6gu.ripple-cask.ru/jid43vpd,1,urlhaus,36,2,1,3,1,0,0
3,http://130.12.180.34/596a96cc7bf9108cd896f33c4...,1,urlhaus,91,4,0,46,0,1,0
4,https://bronze.systems,0,tranco,22,1,0,0,1,0,0


In [11]:
from sklearn.model_selection import train_test_split


features = ["url_length", "dot_count", "hyphen_count", "digit_count", "has_https", "has_ip", "has_suspicious"]

X = df[features]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

print("Train:", X_train.shape, "Test:", y_test.shape)



Train: (39049, 7) Test: (9763,)


In [12]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state = 42)
model.fit(X_train, y_train)

print("Model trained.")


Model trained.


In [13]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy: ", accuracy)

Accuracy:  0.9976441667520229


In [14]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[4781    5]
 [  18 4959]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4786
           1       1.00      1.00      1.00      4977

    accuracy                           1.00      9763
   macro avg       1.00      1.00      1.00      9763
weighted avg       1.00      1.00      1.00      9763



In [15]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

def explain_url(url, features):
    prompt = f""" You are a security researcher.

Analyse following URL in terms of phishing
URL: {url}

Features:

-URL length: {features['url_length']}
-Digits: {features['digit_count']}
-uses IP instead of domain: {'yes' if features['has_ip'] else 'no'}
-uses https: {'yes' if features['has_https'] else 'no'}
-dots: {features['dot_count']}

Explain shortly why following URL might be phishing"""
    llm = genai.GenerativeModel("gemini-2.5-flash-lite")
    response = llm.generate_content(prompt)
    return response.text
    
row = df[df['label'] == 1].iloc[0]
print("URL:", row['url'])
print(explain_url(row['url'], row))

c:\Users\Krzychu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Krzychu\AppData\Local\Temp\ipykernel_20884\2469969901.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


URL: http://110.37.26.193:54956/bin.sh
As a security researcher, I can analyze the provided URL for potential phishing indicators.

**URL Analysis:** `http://110.37.26.193:54956/bin.sh`

**Features:**

*   **URL length:** 33 (Relatively short, but not inherently suspicious on its own)
*   **Digits:** 15 (A significant number of digits, which can be a red flag when combined with an IP address)
*   **Uses IP instead of domain:** Yes (This is a **major red flag**)
*   **Uses https:** No (Missing HTTPS, meaning the connection is **unencrypted**, a significant security risk and common in phishing)
*   **Dots:** 4 (Expected for an IP address)

**Short Explanation Why This URL Might Be Phishing:**

This URL exhibits several strong indicators of phishing:

1.  **IP Address Instead of Domain:** Legitimate websites almost always use domain names (e.g., `google.com`, `bankofamerica.com`). Using a raw IP address is highly unusual for legitimate services and is a common tactic in phishing to bypass